In [ ]:
from keras.applications.vgg16 import preprocess_input
from keras.models import Model
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import cv2
from keras.preprocessing import image
from keras.applications.vgg16 import VGG16, preprocess_input, decode_predictions
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
import tensorflow_hub as hub
import os

In [ ]:
class TransferStyleVideo(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.model = hub.load('https://tfhub.dev/google/magenta/arbitrary-image-stylization-v1-256/2')

    def load_image(self, image_path):
        img = plt.imread(image_path)
        img = img.astype(np.float32)[np.newaxis, ...] / 255.
        return img

    def stylize_image(self, content_image, style_image):
        style_image = tf.image.resize(style_image, (256, 256))
        stylized_image = self.model(tf.constant(content_image), tf.constant(style_image))[0]
        return stylized_image

    def predict(self, X, Y):
        content_image = self.load_image(X)
        style_image = self.load_image(Y)
        stylized_image = self.stylize_image(content_image, style_image)
        return stylized_image

    def transfer_style(self, X, Y, video):
        style_image = self.load_image(Y)
        cap = cv2.VideoCapture(video)

        frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = int(cap.get(cv2.CAP_PROP_FPS))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        output_video_path = '/content/output_video_transfer_style.mp4'
        out = cv2.VideoWriter(output_video_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame_width, frame_height))

        frame_count = 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = frame.astype(np.float32)[np.newaxis, ...] / 255.

            # Apply style transfer
            stylized_frame = self.stylize_image(frame, style_image)
            stylized_frame = np.squeeze(stylized_frame)
            stylized_frame = (stylized_frame * 255).astype(np.uint8)
            stylized_frame = cv2.cvtColor(stylized_frame, cv2.COLOR_RGB2BGR)

            # Write the frame to the output video
            out.write(stylized_frame)

            frame_count += 1
            if frame_count % 10 == 0:
                print(f'Processed {frame_count}/{total_frames} frames')

        cap.release()
        out.release()

        # Проверка существования файла и его размеров
        if os.path.exists(output_video_path):
            print(f'Video processing complete. Saved to: {output_video_path}')
            print(f'File size: {os.path.getsize(output_video_path)} bytes')
        else:
            print('Video file not found after processing')

        return output_video_path

# Создаем конвейер
pipe = Pipeline(steps=[
    ('transfer_style', TransferStyleVideo())
])

# Применяем стиль к видео
output_video_path = pipe.named_steps['transfer_style'].transfer_style('/content/Hardy-1.jpg', '/content/st-steam.jpg', '/content/Di-4.mp4')

print(f'Video saved to: {output_video_path}')

Processed 10/165 frames
Processed 20/165 frames
Processed 30/165 frames
Processed 40/165 frames
Processed 50/165 frames
Processed 60/165 frames
Processed 70/165 frames
Processed 80/165 frames
Processed 90/165 frames
Processed 100/165 frames
Processed 110/165 frames
Processed 120/165 frames
Processed 130/165 frames
Processed 140/165 frames
Processed 150/165 frames
Processed 160/165 frames


NameError: name 'os' is not defined